In [2]:
import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

# ============================================================
# STEP 1: Load data
# ============================================================
df = pd.read_csv('annotations_indoor_outdoor.csv')

print(f'Images in sample: {len(df)}')

# Nur vollständig annotierte Fälle
df = df.dropna(subset=['annotator_NB', 'annotator_LE'])

print(f'Fully labelled images: {len(df)}')

# ============================================================
# STEP 2: Cohen's Kappa
# ============================================================
kappa = cohen_kappa_score(
    df['annotator_NB'],
    df['annotator_LE']
)

agreement = (
    df['annotator_NB'] == df['annotator_LE']
).mean()

print(f"\nCohen's Kappa (NB vs LE): {kappa:.3f}")
print(f"Raw Agreement (NB vs LE): {agreement:.1%}")

# ============================================================
# STEP 3: Confusion Matrix
# ============================================================
labels = sorted(df['annotator_NB'].unique())

cm = confusion_matrix(
    df['annotator_NB'],
    df['annotator_LE'],
    labels=labels
)

print('\nConfusion Matrix (Rows=NB, Cols=LE):')
print(pd.DataFrame(cm, index=labels, columns=labels))

# ============================================================
# STEP 4: Save disagreements
# ============================================================
df['agreement'] = (
    df['annotator_NB'] == df['annotator_LE']
)

disagreements = df[~df['agreement']]

print(f'\nImages with disagreement: {len(disagreements)} of {len(df)}')

disagreements.to_csv(
    'disagreements_to_reconcile_indoor_outdoor.csv',
    index=False
)

print('✅ disagreements_to_reconcile_indoor_outdoor.csv saved')

# ============================================================
# STEP 5: Save metrics summary
# ============================================================
summary = pd.DataFrame({
    'Metric': [
        "Cohen's Kappa (NB vs LE)",
        "Raw Agreement (NB vs LE)",
        "Images with disagreement",
        "Total Images"
    ],
    'Value': [
        round(kappa, 3),
        f'{agreement:.1%}',
        len(disagreements),
        len(df)
    ]
})

summary.to_csv(
    'agreement_metrics_indoor_outdoor.csv',
    index=False
)

print('✅ agreement_metrics_indoor_outdoor.csv saved')

Images in sample: 300
Fully labelled images: 300

Cohen's Kappa (NB vs LE): 0.813
Raw Agreement (NB vs LE): 98.0%

Confusion Matrix (Rows=NB, Cols=LE):
         indoor  outdoor
indoor       14        5
outdoor       1      280

Images with disagreement: 6 of 300
✅ disagreements_to_reconcile_indoor_outdoor.csv saved
✅ agreement_metrics_indoor_outdoor.csv saved
